In [ ]:
# import depsi
import xarray as xr
from pathlib import Path
from depsi.io import read_slc_stack
from depsi.classification import ps_selection, network_stm_selection

In [ ]:
# # Small data ~70MB
# path_stack = Path('/project/caroline/Share/stacks_zarr/nl_amsterdam_s1_asc_t161/nl_amsterdam_s1_asc_t161.zarr')

In [ ]:
# Large data ~4.4GB
path_stack = Path('/project/caroline/Share/stacks_zarr/nl_assendelft_s1_dsc_t037/nl_assendelft_s1_dsc_t037.zarr/')

# Using a Dask SLURM cluster with 6 workers = 24 cores + 192GB mem
from dask.distributed import Client
client = Client("tcp://10.0.2.69:44685")
client

2026-02-13 09:45:56,721 - distributed.client - ERROR - Failed to reconnect to scheduler after 30.00 seconds, closing client


In [ ]:
%%time
# Lazy loading stack
stack = read_slc_stack(path_stack)
stack = stack.compute()

In [ ]:
%%time
# PS selection
stm_preselention_pnts = ps_selection(
    stack,
    method="nmad",
    threshold=0.5,  # Points with NMAD below this threshold are selected
    output_chunks=2500,
    mem_persist=False,
)
print(f"{stm_preselention_pnts.sizes["space"]} preselction points selected")


# Network pnts selection
mask = stm_preselention_pnts["full_ts_nmad"] < 0.2
stm_network_pnts = stm_preselention_pnts.where(mask.compute(), drop=True)
stm_network_pnts = network_stm_selection(
    stm_network_pnts,
    min_dist=100,
    azimuth_spacing=10,
    range_spacing=5,
    sortby_var="full_ts_nmad",  # The sort is in ascending order so that points with smaller NMAD are prioritized
)
print(f"{stm_network_pnts.sizes["space"]}preselction points selected")

In [ ]:
%%time
# Write preselection and network points to zarr
stm_preselention_pnts.to_zarr("./stm_preselection.zarr", mode="w")
stm_network_pnts.to_zarr("./stm_network_pnts.zarr", mode="w")

small dataset:

```
loading: 2min 16s
processing: 2.61s
writing: 12ms
```

large dataset:

```
loading: 35.5s
processing: 2min25s
writing: 12ms
```